# Phase 10 — MLflow Tracking
## Real Estate Investment Advisor

This notebook tracks the **already-trained** classification and regression models.

### Classification
- Logistic Regression
- Decision Tree
- Random Forest
- Extra Trees
- XGBoost

### Regression
- Linear Regression
- Decision Tree
- Random Forest
- Extra Trees
- XGBoost

**Important:** This version uses a SQLite MLflow backend instead of the deprecated filesystem tracking backend (`mlruns`).

In [ ]:
import os
import joblib
import pandas as pd
import mlflow
import mlflow.sklearn

print("MLflow version:", mlflow.__version__)
print("Python version:", __import__("sys").version.split()[0])

## 1. Configure MLflow with SQLite

In [ ]:
# Project root is one level above the notebooks folder
PROJECT_ROOT = os.path.abspath("..")

# SQLite database for MLflow experiment/run metadata
MLFLOW_DB = os.path.join(PROJECT_ROOT, "mlflow.db")

# Local folder for MLflow model artifacts
MLFLOW_ARTIFACTS = os.path.join(PROJECT_ROOT, "mlartifacts")
os.makedirs(MLFLOW_ARTIFACTS, exist_ok=True)

# IMPORTANT:
# Use SQLite for MLflow tracking metadata.
# Do NOT use ../mlruns as the tracking URI.
tracking_uri = "sqlite:///" + MLFLOW_DB.replace("\\", "/")

mlflow.set_tracking_uri(tracking_uri)

print("MLflow tracking URI:")
print(mlflow.get_tracking_uri())

print("\nMLflow database:")
print(MLFLOW_DB)

print("\nMLflow artifact directory:")
print(MLFLOW_ARTIFACTS)

In [ ]:
# Initialize the MLflow client after setting the tracking URI

from mlflow.tracking import MlflowClient

client = MlflowClient()

print("MLflow SQLite backend initialized successfully.")

## 2. Verify Required Model Files

In [ ]:
required_files = [
    "../models/classification_results.csv",
    "../models/logistic_regression.pkl",
    "../models/decision_tree.pkl",
    "../models/random_forest.pkl",
    "../models/extra_trees.pkl",
    "../models/xgboost_classifier.pkl",
    "../models/regression_results.csv",
    "../models/linear_regression.pkl",
    "../models/decision_tree_regressor.pkl",
    "../models/random_forest_regressor.pkl",
    "../models/extra_trees_regressor.pkl",
    "../models/xgboost_regressor.pkl",
]

missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print("Missing required files:")
    for file in missing_files:
        print(" -", file)
    raise FileNotFoundError(
        "Required model/result files are missing. "
        "Make sure Phase 8 and Phase 9 were completed first."
    )

print("All required model and result files are present.")

## Part A — Classification

In [ ]:
classification_results = pd.read_csv(
    "../models/classification_results.csv"
)

print("Classification results:")
display(classification_results)

In [ ]:
# Load trained classification models

classification_models = {
    "Logistic Regression": joblib.load("../models/logistic_regression.pkl"),
    "Decision Tree": joblib.load("../models/decision_tree.pkl"),
    "Random Forest": joblib.load("../models/random_forest.pkl"),
    "Extra Trees": joblib.load("../models/extra_trees.pkl"),
    "XGBoost": joblib.load("../models/xgboost_classifier.pkl"),
}

print("Loaded classification models:")
for name, model in classification_models.items():
    print(f"- {name}: {type(model).__name__}")

In [ ]:
# Create or retrieve the classification experiment

classification_experiment_name = (
    "Real Estate Investment Advisor - Classification"
)

classification_experiment = mlflow.get_experiment_by_name(
    classification_experiment_name
)

if classification_experiment is None:
    classification_experiment_id = mlflow.create_experiment(
        classification_experiment_name,
        artifact_location="file://" + MLFLOW_ARTIFACTS.replace("\\", "/")
    )
else:
    classification_experiment_id = classification_experiment.experiment_id

print("Classification experiment ID:", classification_experiment_id)

In [ ]:
# Log classification models and their previously calculated metrics

for model_name, model in classification_models.items():

    matching_rows = classification_results[
        classification_results["Model"].astype(str).str.strip() == model_name
    ]

    if matching_rows.empty:
        raise ValueError(
            f"No results row found for classification model: {model_name}"
        )

    result = matching_rows.iloc[0]

    with mlflow.start_run(
        experiment_id=classification_experiment_id,
        run_name=model_name
    ):
        mlflow.log_param("model_type", model_name)

        mlflow.log_metric("accuracy", float(result["Accuracy"]))
        mlflow.log_metric("precision", float(result["Precision"]))
        mlflow.log_metric("recall", float(result["Recall"]))
        mlflow.log_metric("f1_score", float(result["F1 Score"]))
        mlflow.log_metric("roc_auc", float(result["ROC AUC"]))

        # XGBoost has its own MLflow flavor.
        if model_name == "XGBoost":
            try:
                import mlflow.xgboost
                mlflow.xgboost.log_model(
                    model,
                    name="model"
                )
            except Exception:
                # Fallback to the sklearn flavor if the installed
                # MLflow/XGBoost combination does not support the
                # native XGBoost flavor.
                mlflow.sklearn.log_model(
                    model,
                    name="model"
                )
        else:
            mlflow.sklearn.log_model(
                model,
                name="model"
            )

        print(f"Logged: {model_name}")

In [ ]:
# Verify classification runs

classification_runs = mlflow.search_runs(
    experiment_ids=[classification_experiment_id],
    order_by=["start_time DESC"]
)

classification_columns = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score",
    "metrics.roc_auc"
]

display(classification_runs[classification_columns])

## Part B — Regression

In [ ]:
regression_results = pd.read_csv(
    "../models/regression_results.csv"
)

print("Regression results:")
display(regression_results)

In [ ]:
# Load trained regression models

regression_models = {
    "Linear Regression": joblib.load("../models/linear_regression.pkl"),
    "Decision Tree": joblib.load("../models/decision_tree_regressor.pkl"),
    "Random Forest": joblib.load("../models/random_forest_regressor.pkl"),
    "Extra Trees": joblib.load("../models/extra_trees_regressor.pkl"),
    "XGBoost": joblib.load("../models/xgboost_regressor.pkl"),
}

print("Loaded regression models:")
for name, model in regression_models.items():
    print(f"- {name}: {type(model).__name__}")

In [ ]:
# Create or retrieve the regression experiment

regression_experiment_name = (
    "Real Estate Investment Advisor - Regression"
)

regression_experiment = mlflow.get_experiment_by_name(
    regression_experiment_name
)

if regression_experiment is None:
    regression_experiment_id = mlflow.create_experiment(
        regression_experiment_name,
        artifact_location="file://" + MLFLOW_ARTIFACTS.replace("\\", "/")
    )
else:
    regression_experiment_id = regression_experiment.experiment_id

print("Regression experiment ID:", regression_experiment_id)

In [ ]:
# Log regression models and their previously calculated metrics

for model_name, model in regression_models.items():

    matching_rows = regression_results[
        regression_results["Model"].astype(str).str.strip() == model_name
    ]

    if matching_rows.empty:
        raise ValueError(
            f"No results row found for regression model: {model_name}"
        )

    result = matching_rows.iloc[0]

    with mlflow.start_run(
        experiment_id=regression_experiment_id,
        run_name=model_name
    ):
        mlflow.log_param("model_type", model_name)

        mlflow.log_metric("MAE", float(result["MAE"]))
        mlflow.log_metric("MSE", float(result["MSE"]))
        mlflow.log_metric("RMSE", float(result["RMSE"]))
        mlflow.log_metric("R2", float(result["R2"]))

        if model_name == "XGBoost":
            try:
                import mlflow.xgboost
                mlflow.xgboost.log_model(
                    model,
                    name="model"
                )
            except Exception:
                mlflow.sklearn.log_model(
                    model,
                    name="model"
                )
        else:
            mlflow.sklearn.log_model(
                model,
                name="model"
            )

        print(f"Logged: {model_name}")

In [ ]:
# Verify regression runs

regression_runs = mlflow.search_runs(
    experiment_ids=[regression_experiment_id],
    order_by=["start_time DESC"]
)

regression_columns = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.MAE",
    "metrics.MSE",
    "metrics.RMSE",
    "metrics.R2"
]

display(regression_runs[regression_columns])

## Part C — Save MLflow Run Summaries

In [ ]:
classification_mlflow_summary = classification_runs[
    classification_columns
].copy()

regression_mlflow_summary = regression_runs[
    regression_columns
].copy()

classification_mlflow_summary.to_csv(
    "../models/mlflow_classification_runs.csv",
    index=False
)

regression_mlflow_summary.to_csv(
    "../models/mlflow_regression_runs.csv",
    index=False
)

print("Saved:")
print("- ../models/mlflow_classification_runs.csv")
print("- ../models/mlflow_regression_runs.csv")

In [ ]:
# Final verification

print("=" * 65)
print("PHASE 10 — MLFLOW TRACKING COMPLETE")
print("=" * 65)

print("Tracking URI:")
print(mlflow.get_tracking_uri())

print("\nClassification experiment:")
print(classification_experiment_name)
print("Runs found:", len(classification_runs))

print("\nRegression experiment:")
print(regression_experiment_name)
print("Runs found:", len(regression_runs))

print("\nMLflow database:")
print(MLFLOW_DB)

print("\nMLflow artifacts:")
print(MLFLOW_ARTIFACTS)

print("\nSummary files:")
print("../models/mlflow_classification_runs.csv")
print("../models/mlflow_regression_runs.csv")

print("\nSUCCESS — Phase 10 MLflow tracking completed.")